In [ ]:
import os
import ast
import random
import numpy as np
import pandas as pd

DATA_DIR = "/content/snuaichallenge_data"

TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TRAIN_IMAGE_DIR = os.path.join(DATA_DIR, "train")

OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "SNU_AI_Challenge/qwen2vl_multitask_v1"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

assert os.path.exists(TRAIN_CSV)
assert os.path.isdir(TRAIN_IMAGE_DIR)

train_df = pd.read_csv(TRAIN_CSV)
train_df["Id"] = train_df["Id"].astype(str)


def parse_answer(answer):
    """CSV의 Answer 문자열을 [1, 2, 3, 4] 리스트로 변환."""
    if isinstance(answer, list):
        result = answer
    else:
        result = ast.literal_eval(str(answer))

    result = [int(value) for value in result]

    if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
        raise ValueError(f"잘못된 Answer: {answer}")

    return result


train_df["Answer_list"] = train_df["Answer"].apply(parse_answer)

print("전체 데이터 수:", len(train_df))
display(
    train_df[
        [
            "Id",
            "Sentence",
            "Input_1",
            "Input_2",
            "Input_3",
            "Input_4",
            "Answer_list"
        ]
    ].head()
)

In [ ]:
SEED = 42
VALID_RATIO = 0.1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

unique_ids = train_df["Id"].unique().copy()

rng = np.random.default_rng(SEED)
rng.shuffle(unique_ids)

valid_size = max(1, int(len(unique_ids) * VALID_RATIO))

valid_ids = set(unique_ids[:valid_size])
training_ids = set(unique_ids[valid_size:])

training_df = (
    train_df[train_df["Id"].isin(training_ids)]
    .reset_index(drop=True)
)

validation_df = (
    train_df[train_df["Id"].isin(valid_ids)]
    .reset_index(drop=True)
)

print("학습 원본 영상 수:", len(training_df))
print("검증 원본 영상 수:", len(validation_df))

assert set(training_df["Id"]).isdisjoint(
    set(validation_df["Id"])
)

In [ ]:
from torch.utils.data import Dataset


class FrameOrderMultiTaskDataset(Dataset):
    """
    ORDER:
        네 이미지 각각의 실제 시간 순위를 예측

    PAIRWISE:
        이미지 A와 B 중 어느 것이 먼저인지 예측

    RANK:
        특정 Input 이미지의 실제 시간 순위를 예측
    """

    def __init__(
        self,
        dataframe,
        image_root,
        tasks=("ORDER", "PAIRWISE", "RANK"),
        augment=True,
        seed=42
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_root = image_root
        self.tasks = tuple(tasks)
        self.augment = augment
        self.seed = seed

    def __len__(self):
        return len(self.dataframe) * len(self.tasks)

    def _get_rng(self, index):
        # Train은 접근할 때마다 다르게 섞음
        if self.augment:
            return random

        # Validation은 항상 같은 문제 생성
        return random.Random(self.seed + index)

    def __getitem__(self, index):
        task_count = len(self.tasks)

        row_index = index // task_count
        task = self.tasks[index % task_count]

        row = self.dataframe.iloc[row_index]
        sample_id = str(row["Id"])

        sentence = (
            ""
            if pd.isna(row["Sentence"])
            else str(row["Sentence"])
        )

        image_paths = [
            os.path.join(
                self.image_root,
                sample_id,
                str(row[f"Input_{i}"])
            )
            for i in range(1, 5)
        ]

        temporal_ranks = [
            int(value)
            for value in row["Answer_list"]
        ]

        for image_path in image_paths:
            if not os.path.exists(image_path):
                raise FileNotFoundError(image_path)

        local_rng = self._get_rng(index)

        # 학습할 때마다 이미지 입력 순서를 무작위로 변경
        if self.augment:
            permutation = list(range(4))
            local_rng.shuffle(permutation)

            image_paths = [
                image_paths[i]
                for i in permutation
            ]

            # 이미지에 붙어 있던 실제 시간 순위도 함께 이동
            temporal_ranks = [
                temporal_ranks[i]
                for i in permutation
            ]

        if task == "ORDER":
            image_labels = [
                "Input image 1",
                "Input image 2",
                "Input image 3",
                "Input image 4"
            ]

            instruction = (
                f'The video is described as: "{sentence}"\n\n'
                "The four images are shuffled frames from the video. "
                "Compare the visual states and temporal progression.\n"
                "Return the actual temporal rank of Input image 1, "
                "Input image 2, Input image 3, and Input image 4 "
                "in that order.\n"
                "Respond only with one Python-style permutation list."
            )

            target = str(
                [int(value) for value in temporal_ranks]
            )

        elif task == "PAIRWISE":
            first_index, second_index = local_rng.sample(
                range(4),
                2
            )

            selected_paths = [
                image_paths[first_index],
                image_paths[second_index]
            ]

            first_rank = temporal_ranks[first_index]
            second_rank = temporal_ranks[second_index]

            image_paths = selected_paths
            image_labels = ["Image A", "Image B"]

            instruction = (
                f'The video is described as: "{sentence}"\n\n'
                "Determine which of the two frames occurs earlier "
                "in the original video.\n"
                "Respond only with A or B."
            )

            target = (
                "A"
                if first_rank < second_rank
                else "B"
            )

        elif task == "RANK":
            selected_index = local_rng.randrange(4)

            image_labels = [
                "Input image 1",
                "Input image 2",
                "Input image 3",
                "Input image 4"
            ]

            instruction = (
                f'The video is described as: "{sentence}"\n\n'
                "The four images are shuffled frames from the video. "
                f"What is the actual temporal rank of "
                f"Input image {selected_index + 1}?\n"
                "Respond only with one integer from 1 through 4."
            )

            target = str(
                int(temporal_ranks[selected_index])
            )

        else:
            raise ValueError(f"지원하지 않는 task: {task}")

        return {
            "Id": sample_id,
            "task": task,
            "image_paths": image_paths,
            "image_labels": image_labels,
            "instruction": instruction,
            "target": target
        }

In [ ]:
### train/validation dataset 분리

In [ ]:
training_dataset = FrameOrderMultiTaskDataset(
    dataframe=training_df,
    image_root=TRAIN_IMAGE_DIR,
    tasks=("ORDER",),
    augment=True,
    seed=SEED
)

validation_dataset = FrameOrderMultiTaskDataset(
    dataframe=validation_df,
    image_root=TRAIN_IMAGE_DIR,
    tasks=("ORDER",),
    augment=False,
    seed=SEED
)

print("학습 예시 수:", len(training_dataset))
print("검증 예시 수:", len(validation_dataset))

for index in range(3):
    example = training_dataset[index]

    print("=" * 60)
    print("Task:", example["task"])
    print("Target:", example["target"])
    print("Images:", example["image_paths"])

In [ ]:
%pip install -q -U "bitsandbytes>=0.46.1"

In [ ]:
### 모델 로드

In [ ]:
import torch

from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen2VLForConditionalGeneration
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)


MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.config.use_cache = False

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "v_proj"
    ],
    task_type="CAUSAL_LM"
)

model = get_peft_model(
    model,
    lora_config
)

# 혹시 Vision Encoder 쪽에 LoRA가 붙었으면 고정
for parameter_name, parameter in model.named_parameters():
    lower_name = parameter_name.lower()

    if (
        "visual" in lower_name
        or "vision" in lower_name
    ):
        parameter.requires_grad = False

model.print_trainable_parameters()

### 학습 파라미터 모듈 수 체크

In [ ]:
trainable_parameter_names = [
    name
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
]

print("학습 파라미터 모듈 수:", len(trainable_parameter_names))

for name in trainable_parameter_names[:30]:
    print(name)

### 학습 코드
SMOKE_TEST에서 max_steps를 조절해서 전체 학습 전 테스트 가능

In [ ]:
import inspect

from transformers import (
    Trainer,
    TrainingArguments
)

SMOKE_TEST = False

if SMOKE_TEST:
    from torch.utils.data import Subset

    actual_training_dataset = Subset(
        training_dataset,
        range(min(30, len(training_dataset)))
    )

    actual_validation_dataset = Subset(
        validation_dataset,
        range(min(10, len(validation_dataset)))
    )

    max_steps = 100
else:
    actual_training_dataset = training_dataset
    actual_validation_dataset = validation_dataset

    max_steps = -1


training_argument_values = {
    "output_dir": OUTPUT_DIR,
    "num_train_epochs": 1,
    "max_steps": max_steps,

    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 8,

    "learning_rate": 1e-4,
    "warmup_ratio": 0.03,
    "max_grad_norm": 0.3,

    "fp16": True,
    "bf16": False,

    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {
        "use_reentrant": False
    },

    "optim": "paged_adamw_8bit",
    "eval_strategy":"no",

    "logging_steps": 20,

    "save_strategy": "steps",
    "save_steps": 100,
    "save_total_limit": 2,

    "report_to": "none",

    # 커스텀 데이터가 Trainer에 의해 제거되지 않게 함
    "remove_unused_columns": False,

    # PIL 이미지 로딩 충돌 방지
    "dataloader_num_workers": 0,

    "seed": SEED,
    "data_seed": SEED
}

# Transformers 버전에 따라 이름이 달라질 수 있음
training_argument_signature = inspect.signature(
    TrainingArguments.__init__
).parameters

if "eval_strategy" in training_argument_signature:
    training_argument_values["eval_strategy"] = "steps"
else:
    training_argument_values["evaluation_strategy"] = "steps"

training_argument_values["eval_steps"] = (
    5 if SMOKE_TEST else 100
)

training_args = TrainingArguments(
    **training_argument_values
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=actual_training_dataset,
    eval_dataset=actual_validation_dataset,
    data_collator=data_collator
)

trainer.train()